In [ ]:
from pyspark.sql import SparkSession
import os
import sys
from pyspark.sql import SparkSession

# Wskazanie Sparkowi dokładnie tego Pythona, w którym działa wirtualne środowisko
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Test") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.driver.memory", "2g") \
    .master("local[*]") \
    .getOrCreate()
print(spark.version)

c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [2]:
import pyarrow
print(f"Zainstalowana wersja PyArrow: {pyarrow.__version__}")

Zainstalowana wersja PyArrow: 25.0.1


In [5]:
arrow_enabled = spark.conf.get("spark.sql.execution.arrow.pyspark.enabled", "false")
print(f"PyArrow włączony: {arrow_enabled}")

PyArrow włączony: true


In [4]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [6]:
from opensky_api import OpenSkyApi, TokenManager

In [7]:
with OpenSkyApi(token_manager=TokenManager.from_json_file("credentials.json")) as api:
    states = api.get_states()

In [8]:
states_df = states

icao24: str - ICAO24 address of the transmitter in hex string representation.

callsign: str - callsign of the vehicle. Can be None if no callsign has been received.

origin_country: str - inferred through the ICAO24 address.

time_position: int - seconds since epoch of last position report. Can be None if there was no position report received by OpenSky within 15s before.

last_contact: int - seconds since epoch of last received message from this transponder.

longitude: float - in ellipsoidal coordinates (WGS-84) and degrees. Can be None.

latitude: float - in ellipsoidal coordinates (WGS-84) and degrees. Can be None.

geo_altitude: float - geometric altitude in meters. Can be None.

on_ground: bool - true if aircraft is on ground (sends ADS-B surface position reports).

velocity: float - over ground in m/s. Can be None if information not present.

true_track: float - in decimal degrees (0 is north). Can be None if information not present.

vertical_rate: float - in m/s, incline is positive, decline negative. Can be None if information not present.

sensors: list [int] - serial numbers of sensors which received messages from the vehicle within the validity period of this state vector. Can be None if no filtering for sensor has been requested.

baro_altitude: float - barometric altitude in meters. Can be None.

squawk: str - transponder code aka Squawk. Can be None.

spi: bool - special purpose indicator.

position_source: int - origin of this state’s position: 0 = ADS-B, 1 = ASTERIX, 2 = MLAT, 3 = FLARM

category: int - aircraft category: 0 = No information at all, 1 = No ADS-B Emitter Category Information, 2 = Light (< 15500 lbs), 3 = Small (15500 to 75000 lbs), 4 = Large (75000 to 300000 lbs), 5 = High Vortex Large (aircraft such as B-757), 6 = Heavy (> 300000 lbs), 7 = High Performance (> 5g acceleration and 400 kts), 8 = Rotorcraft, 9 = Glider / sailplane, 10 = Lighter-than-air, 11 = Parachutist / Skydiver, 12 = Ultralight / hang-glider / paraglider, 13 = Reserved, 14 = Unmanned Aerial Vehicle, 15 = Space / Trans-atmospheric vehicle, 16 = Surface Vehicle – Emergency Vehicle, 17 = Surface Vehicle – Service Vehicle, 18 = Point Obstacle (includes tethered balloons), 19 = Cluster Obstacle, 20 = Line Obstacle.

In [20]:
lista={}
for state in states.states:
    # lista[state.icao24]['icao24'] = state.icao24
    lista[state.icao24] = {}
    lista[state.icao24]['callsign'] = state.callsign
    lista[state.icao24]['origin_country'] = state.origin_country
    lista[state.icao24]['time_position'] = state.time_position
    lista[state.icao24]['last_contact'] = state.last_contact
    lista[state.icao24]['longitude'] = state.longitude
    lista[state.icao24]['latitude'] = state.latitude
    lista[state.icao24]['geo_altitude'] = state.geo_altitude
    lista[state.icao24]['on_ground'] = state.on_ground
    lista[state.icao24]['velocity'] = state.velocity
    lista[state.icao24]['true_track'] = state.true_track
    lista[state.icao24]['vertical_rate'] = state.vertical_rate
    lista[state.icao24]['sensors'] = state.sensors
    lista[state.icao24]['baro_altitude'] = state.baro_altitude
    lista[state.icao24]['squawk'] = state.squawk
    lista[state.icao24]['spi'] = state.spi
    lista[state.icao24]['position_source'] = state.position_source
    lista[state.icao24]['category'] = state.category

In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, MapType
schema = StructType([
StructField("icao24", StringType(), True),
StructField("callsign",  StringType(), True),
StructField("origin_country",  StringType(), True),
StructField("time_position",  StringType(), True),
StructField("last_contact",  StringType(), True),
StructField("longitude",  StringType(), True),
StructField("latitude",  StringType(), True),
StructField("geo_altitude",  StringType(), True),
StructField("on_ground",  StringType(), True),
StructField("velocity",  StringType(), True),
StructField("true_track",  StringType(), True),
StructField("vertical_rate",  StringType(), True),
StructField("sensors",  StringType(), True),
StructField("baro_altitude",  StringType(), True),
StructField("squawk",  StringType(), True),
StructField("spi",  StringType(), True),
StructField("position_source",  StringType(), True),
StructField("category",  StringType(), True)
])


In [23]:
# print(Poland_aircraft)
aircraft_df=[]
for aircraft in lista:
    aircraft_df.extend(
    [(
        aircraft,
        lista[aircraft]['callsign'],
        lista[aircraft]['origin_country'], 
        lista[aircraft]['time_position'],
        lista[aircraft]['last_contact'],
        lista[aircraft]['longitude'],
        lista[aircraft]['latitude'],
        lista[aircraft]['geo_altitude'],
        lista[aircraft]['on_ground'],
        lista[aircraft]['velocity'],
        lista[aircraft]['true_track'],
        lista[aircraft]['vertical_rate'],
        lista[aircraft]['sensors'],
        lista[aircraft]['baro_altitude'],
        lista[aircraft]['squawk'],
        lista[aircraft]['spi'],
        lista[aircraft]['position_source'],
        lista[aircraft]['category']
        )])
    
    print(f"ICAO24: {aircraft}")
    print(lista[aircraft]['callsign'])
    print(lista[aircraft]['origin_country'])
    print(lista[aircraft]['time_position'])
    print(lista[aircraft]['last_contact'])
    print(lista[aircraft]['longitude'])
    print(lista[aircraft]['latitude'])
    print(lista[aircraft]['geo_altitude'])
    print(lista[aircraft]['on_ground'])
    print(lista[aircraft]['velocity'])
    print(lista[aircraft]['true_track'])
    print(lista[aircraft]['vertical_rate'])
    print(lista[aircraft]['sensors'])
    print(lista[aircraft]['baro_altitude'])
    print(lista[aircraft]['squawk'])
    print(lista[aircraft]['spi'])
    print(lista[aircraft]['position_source'])
    print(lista[aircraft]['category'])
    
    
Poland_aircraft_df = spark.createDataFrame(aircraft_df, schema=schema)

ICAO24: 39de4e
TVF13NV 
France
1786611077
1786611077
2.4911
48.7422
1280.16
False
93.65
88.74
5.2
None
1104.9
7616
False
0
0
ICAO24: e8027c
LPE2428 
Chile
1786610913
1786610913
-46.4929
-23.4404
792.48
False
73.42
73.72
-3.25
None
746.76
None
False
0
0
ICAO24: 80162c
AXB324  
India
1786611077
1786611077
55.6275
24.9803
4686.3
False
163.57
120.84
15.28
None
4488.18
6225
False
0
0
ICAO24: 39de4b
TVF26XF 
France
1786611076
1786611076
30.6259
36.5049
1333.5
False
114.23
187.77
0
None
1272.54
0676
False
0
0
ICAO24: 39de4a
TVF6427 
France
1786611077
1786611077
11.0371
36.5983
11292.84
False
228.75
3.09
3.25
None
10637.52
6161
False
0
0
ICAO24: 39de57
TVF739F 
France
1786611076
1786611076
4.6706
43.1491
12260.58
False
228.23
3.1
-0.33
None
11590.02
1000
False
0
0
ICAO24: 39de59
TVF30VF 
France
1786611077
1786611077
1.4014
48.0467
4594.86
False
165.71
29.99
-7.8
None
4267.2
1000
False
0
0
ICAO24: 801645
AIC1704 
India
1786610926
1786610926
80.8476
26.1575
11430
False
219.94
153.32
0
None
10668

In [24]:
Poland_aircraft_df.show()

+------+--------+--------------+-------------+------------+---------+--------+------------+---------+--------+----------+-------------+-------+-------------+------+-----+---------------+--------+
|icao24|callsign|origin_country|time_position|last_contact|longitude|latitude|geo_altitude|on_ground|velocity|true_track|vertical_rate|sensors|baro_altitude|squawk|  spi|position_source|category|
+------+--------+--------------+-------------+------------+---------+--------+------------+---------+--------+----------+-------------+-------+-------------+------+-----+---------------+--------+
|39de4e|TVF13NV |        France|   1786611077|  1786611077|   2.4911| 48.7422|     1280.16|    false|   93.65|     88.74|          5.2|   NULL|       1104.9|  7616|false|              0|       0|
|e8027c|LPE2428 |         Chile|   1786610913|  1786610913| -46.4929|-23.4404|      792.48|    false|   73.42|     73.72|        -3.25|   NULL|       746.76|  NULL|false|              0|       0|
|80162c|AXB324  |   

In [25]:
Poland_aircraft_df.explain(True)

== Parsed Logical Plan ==
LogicalRDD [icao24#1780, callsign#1781, origin_country#1782, time_position#1783, last_contact#1784, longitude#1785, latitude#1786, geo_altitude#1787, on_ground#1788, velocity#1789, true_track#1790, vertical_rate#1791, sensors#1792, baro_altitude#1793, squawk#1794, spi#1795, position_source#1796, category#1797], false

== Analyzed Logical Plan ==
icao24: string, callsign: string, origin_country: string, time_position: string, last_contact: string, longitude: string, latitude: string, geo_altitude: string, on_ground: string, velocity: string, true_track: string, vertical_rate: string, sensors: string, baro_altitude: string, squawk: string, spi: string, position_source: string, category: string
LogicalRDD [icao24#1780, callsign#1781, origin_country#1782, time_position#1783, last_contact#1784, longitude#1785, latitude#1786, geo_altitude#1787, on_ground#1788, velocity#1789, true_track#1790, vertical_rate#1791, sensors#1792, baro_altitude#1793, squawk#1794, spi#1795